In [4]:
# =============================================================================
# 셀 1: 유니버스 현황 파악
# 목적: S&P500 유니버스에 어떤 정보가 있는지 확인
# 산출물: 없음 (탐색용)
# =============================================================================

import pandas as pd
from pathlib import Path

# SSOT 경로
QP2_ROOT = Path(r"C:/QP2")
META_DIR = QP2_ROOT / "data" / "meta"

# 유니버스 로드
universe = pd.read_parquet(META_DIR / "sp500_universe.parquet")

# 기본 정보
print("="*60)
print(f"총 종목 수: {len(universe)}")
print(f"컬럼 목록: {universe.columns.tolist()}")
print("="*60)

# 샘플 확인
universe.head(10)

총 종목 수: 503
컬럼 목록: ['ticker_sp', 'ticker_yahoo', 'cik', 'security_name', 'GICS Sector', 'GICS Sub-Industry']


,ticker_sp,ticker_yahoo,cik,security_name,GICS Sector,GICS Sub-Industry
0,MMM,MMM,0000066740,3M,Industrials,Industrial Conglomerates
1,AOS,AOS,0000091142,A. O. Smith,Industrials,Building Products
2,ABT,ABT,0000001800,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,ABBV,0001551152,AbbVie,Health Care,Biotechnology
4,ACN,ACN,0001467373,Accenture,Information Technology,IT Consulting & Other Services
5,ADBE,ADBE,0000796343,Adobe Inc.,Information Technology,Application Software
6,AMD,AMD,0000002488,Advanced Micro Devices,Information Technology,Semiconductors
7,AES,AES,0000874761,AES Corporation,Utilities,Independent Power Producers & Energy Traders
8,AFL,AFL,0000004977,Aflac,Financials,Life & Health Insurance
9,A,A,0001090872,Agilent Technologies,Health Care,Life Sciences Tools & Services


In [5]:
# =============================================================================
# 셀 2: GICS 섹터/산업 분포 확인
# 목적: 어떤 섹터/산업이 있는지 전수 파악
# 산출물: 없음 (탐색용)
# =============================================================================

print("="*60)
print("GICS Sector 분포")
print("="*60)
print(universe['GICS Sector'].value_counts())

print("\n" + "="*60)
print("GICS Sub-Industry 분포 (상위 30개)")
print("="*60)
print(universe['GICS Sub-Industry'].value_counts().head(30))

print(f"\n총 Sub-Industry 종류: {universe['GICS Sub-Industry'].nunique()}개")

GICS Sector 분포
GICS Sector
Industrials               80
Financials                76
Information Technology    70
Health Care               60
Consumer Discretionary    48
Consumer Staples          36
Utilities                 31
Real Estate               31
Materials                 26
Communication Services    23
Energy                    22
Name: count, dtype: int64

GICS Sub-Industry 분포 (상위 30개)
GICS Sub-Industry
Health Care Equipment                             17
Electric Utilities                                15
Application Software                              14
Semiconductors                                    14
Industrial Machinery & Supplies & Components      14
Multi-Utilities                                   12
Asset Management & Custody Banks                  12
Aerospace & Defense                               12
Packaged Foods & Meats                            11
Oil & Gas Exploration & Production                10
Technology Hardware, Storage & Peripherals       

In [8]:
# =============================================================================
# 셀 2-1: GICS Sub-Industry 전체 목록 + 종목 매핑 저장
# 목적: 127개 전체 산업별 종목 현황 파악
# 산출물: data/meta/gics_subindustry_map.txt
# =============================================================================

import pandas as pd
from pathlib import Path

# SSOT 경로
QP2_ROOT = Path(r"C:/QP2")
META_DIR = QP2_ROOT / "data" / "meta"

# 유니버스 로드
universe = pd.read_parquet(META_DIR / "sp500_universe.parquet")

# 출력 파일 경로
OUT_PATH = META_DIR / "gics_subindustry_map.txt"

# 파일 작성
with open(OUT_PATH, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("GICS Sub-Industry 전체 목록 + 소속 종목\n")
    f.write(f"총 {universe['GICS Sub-Industry'].nunique()}개 산업, {len(universe)}개 종목\n")
    f.write("="*80 + "\n\n")
    
    # 섹터별로 그룹핑해서 보기 좋게
    for sector in sorted(universe['GICS Sector'].unique()):
        f.write(f"\n{'#'*80}\n")
        f.write(f"## SECTOR: {sector}\n")
        f.write(f"{'#'*80}\n")
        
        sector_df = universe[universe['GICS Sector'] == sector]
        
        for subind in sorted(sector_df['GICS Sub-Industry'].unique()):
            subind_df = sector_df[sector_df['GICS Sub-Industry'] == subind]
            tickers = subind_df['ticker_yahoo'].tolist()
            names = subind_df['security_name'].tolist()
            
            f.write(f"\n### {subind} ({len(tickers)}개)\n")
            for t, n in zip(tickers, names):
                f.write(f"    {t:6} | {n}\n")

print(f"저장 완료: {OUT_PATH}")
print(f"파일 크기: {OUT_PATH.stat().st_size / 1024:.1f} KB")

저장 완료: C:\QP2\data\meta\gics_subindustry_map.txt
파일 크기: 21.4 KB


In [9]:
# =============================================================================
# 셀 3: Finnhub News API 연결 + 테스트
# 목적: API 작동 확인 + 뉴스 데이터 구조 파악
# 산출물: 없음 (탐색용)
# 주의: API 호출 제한 있음 (무료: 60 calls/min)
# =============================================================================

import requests
import pandas as pd
from datetime import datetime, timedelta

# Finnhub API 키 (네 API_Key.txt에서)
FINNHUB_API_KEY = "d5u2mfpr01qtjet1q8bgd5u2mfpr01qtjet1q8c0"

# 테스트: AAPL 뉴스 가져오기
def get_company_news(ticker: str, days_back: int = 7) -> list:
    """
    특정 종목의 최근 뉴스 가져오기
    """
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days_back)
    
    url = "https://finnhub.io/api/v1/company-news"
    params = {
        "symbol": ticker,
        "from": start_date.strftime("%Y-%m-%d"),
        "to": end_date.strftime("%Y-%m-%d"),
        "token": FINNHUB_API_KEY
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code}")
        return []

# 테스트 실행
news = get_company_news("AAPL", days_back=7)

print(f"AAPL 최근 7일 뉴스: {len(news)}건")
print("="*60)

# 첫 번째 뉴스 구조 확인
if news:
    sample = news[0]
    print("뉴스 데이터 구조:")
    for key, value in sample.items():
        if key == "summary":
            print(f"  {key}: {value[:100]}...")  # 요약은 100자만
        else:
            print(f"  {key}: {value}")

AAPL 최근 7일 뉴스: 249건
뉴스 데이터 구조:
  category: company
  datetime: 1770003961
  headline: US futures, Asian shares slip, tracking Wall Street's retreat, while oil falls more than $2
  id: 138328758
  image: https://s.yimg.com/rz/stage/p/yahoo_finance_en-US_h_p_finance_2.png
  related: AAPL
  source: Yahoo
  summary: U.S. futures and Asian shares skidded Monday and oil prices fell more than $2 a barrel.  In South Ko...
  url: https://finnhub.io/api/news?id=161ac4d38507dcfd279f87adfd2f7a46bd804470ef5b8517f647f8dc67ee6680


In [10]:
# =============================================================================
# 셀 3-1: 뉴스 품질 확인
# 목적: headline에 실제로 AAPL/Apple 언급된 비율 체크
# 산출물: 없음 (탐색용)
# =============================================================================

# headline에 Apple 또는 AAPL 포함된 뉴스 필터링
direct_news = [
    n for n in news 
    if "apple" in n["headline"].lower() or "aapl" in n["headline"].lower()
]

print(f"전체 뉴스: {len(news)}건")
print(f"headline에 Apple/AAPL 직접 언급: {len(direct_news)}건")
print(f"직접 언급 비율: {len(direct_news)/len(news)*100:.1f}%")

print("\n" + "="*60)
print("직접 언급 뉴스 샘플 (최대 5개):")
print("="*60)
for n in direct_news[:5]:
    print(f"- {n['headline'][:80]}...")

print("\n" + "="*60)
print("간접 언급 뉴스 샘플 (최대 5개):")
print("="*60)
indirect_news = [n for n in news if n not in direct_news]
for n in indirect_news[:5]:
    print(f"- {n['headline'][:80]}...")

전체 뉴스: 249건
headline에 Apple/AAPL 직접 언급: 125건
직접 언급 비율: 50.2%

직접 언급 뉴스 샘플 (최대 5개):
- Is It Time to Take a Bite Out of Apple's Stock as Revenue Growth Accelerates?...
- Is Apple (AAPL) Fairly Priced After Recent Share Price Pullback?...
- Apple CEO sends blunt message iPhone 18 fans can’t ignore...
- Why JPMorgan Says Apple’s (AAPL) Results Should Calm Investor Nerves...
- Intel’s Foundry Pivot Deepens With Apple Talks And New U.S. Stake...

간접 언급 뉴스 샘플 (최대 5개):
- US futures, Asian shares slip, tracking Wall Street's retreat, while oil falls m...
- Advisory Firm Slashes Income Fund by $8.4 Million While Nvidia and VTI Lead Hold...
- Westfuller Advisors Doubles Down on Ultra-Short Treasuries With Second Cash ETF...
- Why a $6 Million Credit Fund Bet Makes Sense With a 13% Yield on the Table...
- Warren Buffett Said If He Were 30 Starting Over With $1M, He'd Put It All In A L...


In [11]:
# =============================================================================
# 셀 4: 뉴스 수집 함수 (필터링 포함) + 다중 종목 테스트
# 목적: S&P500 전체 대상 뉴스 수집 파이프라인 구축
# 산출물: 없음 (함수 정의 + 테스트)
# 주의: API 제한 (60 calls/min) → sleep 포함
# =============================================================================

import requests
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
import time

# 설정
FINNHUB_API_KEY = "d5u2mfpr01qtjet1q8bgd5u2mfpr01qtjet1q8c0"
QP2_ROOT = Path(r"C:/QP2")
META_DIR = QP2_ROOT / "data" / "meta"

# 유니버스 로드
universe = pd.read_parquet(META_DIR / "sp500_universe.parquet")

# 회사명 매핑 (티커 → 회사명 키워드)
# 예: AAPL → ["apple", "aapl"]
def get_company_keywords(ticker: str, security_name: str) -> list:
    """회사명에서 검색용 키워드 추출"""
    keywords = [ticker.lower()]
    
    # 회사명에서 핵심 단어 추출 (Inc., Corp. 등 제거)
    name_clean = security_name.lower()
    for remove in ["inc.", "inc", "corp.", "corp", "corporation", "company", 
                   "the", "ltd.", "ltd", "plc", "llc", "&", ","]:
        name_clean = name_clean.replace(remove, " ")
    
    # 첫 번째 단어 (보통 회사 핵심 이름)
    first_word = name_clean.split()[0].strip()
    if len(first_word) > 2:  # 너무 짧은 건 제외
        keywords.append(first_word)
    
    return list(set(keywords))


def get_company_news_filtered(ticker: str, security_name: str, days_back: int = 7) -> list:
    """
    종목별 뉴스 수집 + 직접 언급 필터링
    """
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days_back)
    
    url = "https://finnhub.io/api/v1/company-news"
    params = {
        "symbol": ticker,
        "from": start_date.strftime("%Y-%m-%d"),
        "to": end_date.strftime("%Y-%m-%d"),
        "token": FINNHUB_API_KEY
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        print(f"Error {ticker}: {response.status_code}")
        return []
    
    all_news = response.json()
    
    # 키워드 필터링
    keywords = get_company_keywords(ticker, security_name)
    
    filtered = []
    for n in all_news:
        text = (n.get("headline", "") + " " + n.get("summary", "")).lower()
        if any(kw in text for kw in keywords):
            n["ticker"] = ticker  # 티커 추가
            filtered.append(n)
    
    return filtered


# 테스트: 5개 종목
test_tickers = ["AAPL", "NVDA", "MSFT", "JPM", "JNJ"]

print("테스트: 5개 종목 뉴스 수집 (필터링 적용)")
print("="*60)

for ticker in test_tickers:
    row = universe[universe["ticker_yahoo"] == ticker].iloc[0]
    security_name = row["security_name"]
    
    news = get_company_news_filtered(ticker, security_name, days_back=7)
    print(f"{ticker:5} ({security_name[:20]:20}): {len(news):3}건")
    
    time.sleep(1)  # API 제한 회피

print("="*60)
print("완료")

테스트: 5개 종목 뉴스 수집 (필터링 적용)
AAPL  (Apple Inc.          ): 149건
NVDA  (Nvidia              ): 121건
MSFT  (Microsoft           ): 127건
JPM   (JPMorgan Chase      ):  43건
JNJ   (Johnson & Johnson   ):  26건
완료


In [14]:
# =============================================================================
# 셀 5: FinBERT 설치 및 테스트
# 목적: 금융 특화 감성 분석 모델 세팅
# 산출물: 없음 (테스트용)
# 주의: 첫 실행 시 모델 다운로드 (~400MB)
# =============================================================================

# FinBERT 설치 (첫 실행만)
# 터미널에서: pip install transformers torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# FinBERT 로드 (금융 특화 감성 분석)
print("FinBERT 모델 로딩 중...")
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

print("로딩 완료!")

# 감성 분석 함수
def analyze_sentiment(text: str) -> dict:
    """
    텍스트 감성 분석
    반환: {"label": "positive/negative/neutral", "score": 0~1}
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = torch.softmax(outputs.logits, dim=1)
    labels = ["positive", "negative", "neutral"]
    
    max_idx = probs.argmax().item()
    
    return {
        "label": labels[max_idx],
        "score": probs[0][max_idx].item(),
        "positive": probs[0][0].item(),
        "negative": probs[0][1].item(),
        "neutral": probs[0][2].item()
    }


# 테스트
test_headlines = [
    "Apple reports record-breaking quarterly revenue, beats analyst expectations",
    "FDA rejects Johnson & Johnson's new drug application",
    "Microsoft announces routine software update",
    "Nvidia stock plunges 10% on weak guidance",
    "JPMorgan maintains stable outlook amid market uncertainty"
]

print("\n" + "="*60)
print("감성 분석 테스트")
print("="*60)

for headline in test_headlines:
    result = analyze_sentiment(headline)
    print(f"\n{headline[:60]}...")
    print(f"  → {result['label'].upper():8} (신뢰도: {result['score']:.2%})")
    print(f"     [+]{result['positive']:.2f} [-]{result['negative']:.2f} [0]{result['neutral']:.2f}")

c:\QP2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FinBERT 모델 로딩 중...


c:\QP2\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\이노\.cache\huggingface\hub\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 928.45it/s, Materializing param=classifier.weight]                                    

로딩 완료!

감성 분석 테스트

Apple reports record-breaking quarterly revenue, beats analy...
  → POSITIVE (신뢰도: 93.27%)
     [+]0.93 [-]0.03 [0]0.04

FDA rejects Johnson & Johnson's new drug application...
  → NEGATIVE (신뢰도: 87.08%)
     [+]0.02 [-]0.87 [0]0.11

Microsoft announces routine software update...
  → NEUTRAL  (신뢰도: 65.45%)
     [+]0.03 [-]0.32 [0]0.65

Nvidia stock plunges 10% on weak guidance...
  → NEGATIVE (신뢰도: 96.55%)
     [+]0.01 [-]0.97 [0]0.02

JPMorgan maintains stable outlook amid market uncertainty...
  → POSITIVE (신뢰도: 79.82%)
     [+]0.80 [-]0.03 [0]0.17


In [15]:
# =============================================================================
# 셀 6: 실제 뉴스 데이터에 감성 분석 적용
# 목적: 수집한 뉴스에 FinBERT 감성 분석 + 3개 지표 산출
# 산출물: DataFrame (뉴스 + 감성 점수)
# 주의: 뉴스 많으면 시간 걸림 (tqdm으로 진행률 표시)
# =============================================================================

import pandas as pd
from tqdm import tqdm
from datetime import datetime

def analyze_sentiment_full(text: str) -> dict:
    """
    감성 분석 + 3개 지표 산출
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = torch.softmax(outputs.logits, dim=1)
    
    pos = probs[0][0].item()
    neg = probs[0][1].item()
    neu = probs[0][2].item()
    
    return {
        "positive": pos,
        "negative": neg,
        "neutral": neu,
        "direction": pos - neg,           # 방향성: -1 ~ +1
        "confidence": 1 - neu,            # 확신도: 0 ~ 1
        "clarity": abs(pos - neg)         # 명확도: 0 ~ 1
    }


def process_news_sentiment(news_list: list) -> pd.DataFrame:
    """
    뉴스 리스트 → 감성 분석 적용 → DataFrame 반환
    """
    results = []
    
    for n in tqdm(news_list, desc="감성 분석"):
        # headline + summary 결합 (더 많은 맥락)
        text = n.get("headline", "") + " " + n.get("summary", "")
        
        sentiment = analyze_sentiment_full(text)
        
        results.append({
            "ticker": n.get("ticker", ""),
            "datetime": datetime.fromtimestamp(n.get("datetime", 0)),
            "headline": n.get("headline", ""),
            "source": n.get("source", ""),
            **sentiment
        })
    
    return pd.DataFrame(results)


# 테스트: AAPL 뉴스 감성 분석
print("AAPL 뉴스 감성 분석 시작...")
print("="*60)

# 셀 4에서 수집한 뉴스 사용 (다시 수집)
row = universe[universe["ticker_yahoo"] == "AAPL"].iloc[0]
aapl_news = get_company_news_filtered("AAPL", row["security_name"], days_back=7)

# 감성 분석 적용
df_sentiment = process_news_sentiment(aapl_news)

print("="*60)
print(f"분석 완료: {len(df_sentiment)}건")
print(f"\n기초 통계:")
print(df_sentiment[["direction", "confidence", "clarity"]].describe())

# 극단 케이스 샘플
print("\n" + "="*60)
print("가장 긍정적인 뉴스 (Top 3):")
print("="*60)
top_pos = df_sentiment.nlargest(3, "direction")
for _, row in top_pos.iterrows():
    print(f"[{row['direction']:+.2f}] {row['headline'][:70]}...")

print("\n" + "="*60)
print("가장 부정적인 뉴스 (Top 3):")
print("="*60)
top_neg = df_sentiment.nsmallest(3, "direction")
for _, row in top_neg.iterrows():
    print(f"[{row['direction']:+.2f}] {row['headline'][:70]}...")

print("\n" + "="*60)
print("혼돈 상태 (confidence 高 + clarity 低, Top 3):")
print("="*60)
# 확신은 있는데 방향이 불분명한 케이스
chaos = df_sentiment[(df_sentiment["confidence"] > 0.7) & (df_sentiment["clarity"] < 0.3)]
chaos_sorted = chaos.nsmallest(3, "clarity")
if len(chaos_sorted) > 0:
    for _, row in chaos_sorted.iterrows():
        print(f"[dir:{row['direction']:+.2f} conf:{row['confidence']:.2f} clar:{row['clarity']:.2f}]")
        print(f"  {row['headline'][:65]}...")
else:
    print("혼돈 케이스 없음 (좋은 신호)")

AAPL 뉴스 감성 분석 시작...


감성 분석: 100%|██████████| 149/149 [00:13<00:00, 11.28it/s]

분석 완료: 149건

기초 통계:
        direction  confidence     clarity
count  149.000000  149.000000  149.000000
mean     0.195770    0.679248    0.556791
std      0.637631    0.346851    0.364756
min     -0.966826    0.052494    0.002955
25%     -0.104125    0.261420    0.158205
50%      0.185955    0.876937    0.691561
75%      0.846719    0.968884    0.906536
max      0.941399    0.984798    0.966826

가장 긍정적인 뉴스 (Top 3):
[+0.94] Apple Q1 Earnings Beat Estimates, iPhone Drives Top-Line Growth...
[+0.94] Apple Supplier STMicroelectronics Flags Improving Chip Sales...
[+0.94] Apple iPhone 17 Demand Surges, Revenue Up 16%...

가장 부정적인 뉴스 (Top 3):
[-0.97] Apple Stock Pares Losses in Afternoon Trading...
[-0.97] These Stocks Are Today’s Movers: Apple, Sandisk, Western Digital, KLA ...
[-0.96] Samsung profit triples but sees chip shortage continuing...

혼돈 상태 (confidence 高 + clarity 低, Top 3):
[dir:+0.05 conf:0.85 clar:0.05]
  Apple Earnings on Desk. Options Signal a Sharp Stock Move Ahead...
[dir:+

In [18]:
# =============================================================================
# 셀 7: S&P500 전체 뉴스 수집 + 감성 분석
# 목적: 전체 종목 대상 뉴스 파이프라인 실행
# 산출물: data/interim/news_sentiment.parquet
# 주의: 503종목 × API 호출 → 약 10~15분 소요
# =============================================================================

import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from tqdm import tqdm
import time

# 경로 설정
QP2_ROOT = Path(r"C:/QP2")
META_DIR = QP2_ROOT / "data" / "meta"
INTERIM_DIR = QP2_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# 유니버스 로드
universe = pd.read_parquet(META_DIR / "sp500_universe.parquet")

# 수집 기간 설정
DAYS_BACK = 30  # 최근 30일

# 전체 뉴스 수집 + 감성 분석
all_results = []
failed_tickers = []

print(f"S&P500 {len(universe)}개 종목 뉴스 수집 시작")
print(f"기간: 최근 {DAYS_BACK}일")
print("="*60)

for idx, row in tqdm(universe.iterrows(), total=len(universe), desc="뉴스 수집"):
    ticker = row["ticker_yahoo"]
    security_name = row["security_name"]
    sector = row["GICS Sector"]
    sub_industry = row["GICS Sub-Industry"]
    
    try:
        # 뉴스 수집 (필터링 포함)
        news = get_company_news_filtered(ticker, security_name, days_back=DAYS_BACK)
        
        # 감성 분석
        for n in news:
            text = n.get("headline", "") + " " + n.get("summary", "")
            sentiment = analyze_sentiment_full(text)
            
            all_results.append({
                "ticker": ticker,
                "security_name": security_name,
                "sector": sector,
                "sub_industry": sub_industry,
                "datetime": datetime.fromtimestamp(n.get("datetime", 0)),
                "headline": n.get("headline", ""),
                "source": n.get("source", ""),
                **sentiment
            })
        
        # API 제한 회피 (60 calls/min)
        time.sleep(1.1)
        
    except Exception as e:
        failed_tickers.append({"ticker": ticker, "error": str(e)})
        continue

# DataFrame 변환
df_all = pd.DataFrame(all_results)

print("="*60)
print(f"수집 완료: {len(df_all)}건, {df_all['ticker'].nunique()}개 종목")
print(f"실패: {len(failed_tickers)}개 종목")

# 저장
OUT_PATH = INTERIM_DIR / "news_sentiment.parquet"
df_all.to_parquet(OUT_PATH, index=False)
print(f"\n저장 완료: {OUT_PATH}")

# 기초 통계
print("\n" + "="*60)
print("기초 통계")
print("="*60)
print(f"기간: {df_all['datetime'].min()} ~ {df_all['datetime'].max()}")
print(f"\n종목별 뉴스 건수:")
print(df_all.groupby("ticker").size().describe())
print(f"\n감성 지표 분포:")
print(df_all[["direction", "confidence", "clarity"]].describe())

S&P500 503개 종목 뉴스 수집 시작
기간: 최근 30일


뉴스 수집: 100%|██████████| 503/503 [50:00<00:00,  5.97s/it]  


수집 완료: 22935건, 501개 종목
실패: 0개 종목

저장 완료: C:\QP2\data\interim\news_sentiment.parquet

기초 통계
기간: 2026-01-03 09:00:46 ~ 2026-02-02 14:08:21

종목별 뉴스 건수:
count    501.000000
mean      45.778443
std       40.546872
min        2.000000
25%       18.000000
50%       32.000000
75%       56.000000
max      232.000000
dtype: float64

감성 지표 분포:
          direction    confidence       clarity
count  22935.000000  22935.000000  2.293500e+04
mean       0.161497      0.653171  5.607363e-01
std        0.649765      0.363694  3.658394e-01
min       -0.969656      0.043464  5.513430e-07
25%       -0.161091      0.221222  1.382117e-01
50%        0.133141      0.872264  7.195572e-01
75%        0.828017      0.965896  9.054340e-01
max        0.947613      0.991952  9.696561e-01


In [21]:
# =============================================================================
# 셀 8: 뉴스 감성 → 수익률 예측력 백테스트
# 목적: 감성 점수가 실제로 미래 수익률을 예측하는지 검증
# 산출물: 없음 (분석용)
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats

# 경로
QP2_ROOT = Path(r"C:/QP2")
INTERIM_DIR = QP2_ROOT / "data" / "interim"

# 데이터 로드
news = pd.read_parquet(INTERIM_DIR / "news_sentiment.parquet")
px_wide = pd.read_parquet(INTERIM_DIR / "yahoo_adjclose_wide.parquet")

# 날짜 인덱스 확인 및 정리
if "date" in px_wide.columns:
    px_wide = px_wide.set_index("date")
px_wide.index = pd.to_datetime(px_wide.index)

# 일간 수익률 계산
returns = px_wide.pct_change()

# 뉴스 날짜 정리
news["date"] = pd.to_datetime(news["datetime"].dt.date)

# 일별 종목별 감성 집계 (평균)
daily_sentiment = news.groupby(["date", "ticker"]).agg({
    "direction": "mean",
    "confidence": "mean",
    "clarity": "mean",
    "positive": "mean",
    "negative": "mean",
    "neutral": "mean"
}).reset_index()

print(f"일별 감성 데이터: {len(daily_sentiment)}건")
print(f"기간: {daily_sentiment['date'].min()} ~ {daily_sentiment['date'].max()}")

# 미래 수익률 붙이기 (t+1, t+5)
results = []

for _, row in daily_sentiment.iterrows():
    date = row["date"]
    ticker = row["ticker"]
    
    if ticker not in returns.columns:
        continue
    
    # t+1 수익률 (다음 날)
    try:
        future_dates = returns.index[returns.index > date]
        if len(future_dates) >= 1:
            ret_1d = returns.loc[future_dates[0], ticker]
        else:
            ret_1d = np.nan
            
        # t+5 수익률 (5일 후)
        if len(future_dates) >= 5:
            ret_5d = (1 + returns.loc[future_dates[:5], ticker]).prod() - 1
        else:
            ret_5d = np.nan
            
        results.append({
            **row.to_dict(),
            "ret_1d": ret_1d,
            "ret_5d": ret_5d
        })
    except:
        continue

df = pd.DataFrame(results)
df = df.dropna(subset=["ret_1d", "ret_5d"])

print(f"\n분석 가능 데이터: {len(df)}건")

# =============================================================================
# 상관관계 분석
# =============================================================================
print("\n" + "="*60)
print("감성 지표 vs 미래 수익률 상관관계")
print("="*60)

for sent_col in ["direction", "confidence", "clarity"]:
    corr_1d, p_1d = stats.pearsonr(df[sent_col], df["ret_1d"])
    corr_5d, p_5d = stats.pearsonr(df[sent_col], df["ret_5d"])
    
    print(f"\n{sent_col}:")
    print(f"  vs ret_1d: r={corr_1d:+.4f}, p={p_1d:.4f} {'*' if p_1d < 0.05 else ''}")
    print(f"  vs ret_5d: r={corr_5d:+.4f}, p={p_5d:.4f} {'*' if p_5d < 0.05 else ''}")

# =============================================================================
# 분위수별 수익률 (Quintile Analysis)
# =============================================================================
print("\n" + "="*60)
print("Direction 분위수별 평균 수익률")
print("="*60)

df["direction_q"] = pd.qcut(df["direction"], q=5, labels=["Q1(부정)", "Q2", "Q3", "Q4", "Q5(긍정)"])

quintile_ret = df.groupby("direction_q")[["ret_1d", "ret_5d"]].mean() * 100  # %로 변환

print(quintile_ret.round(3))

# Q5 - Q1 스프레드
q5_ret = df[df["direction_q"] == "Q5(긍정)"]["ret_1d"].mean()
q1_ret = df[df["direction_q"] == "Q1(부정)"]["ret_1d"].mean()
spread = (q5_ret - q1_ret) * 100

print(f"\nQ5-Q1 스프레드 (1일): {spread:+.3f}%p")

일별 감성 데이터: 8045건
기간: 2026-01-03 00:00:00 ~ 2026-02-02 00:00:00

분석 가능 데이터: 5079건

감성 지표 vs 미래 수익률 상관관계

direction:
  vs ret_1d: r=+0.0051, p=0.7188 
  vs ret_5d: r=-0.0085, p=0.5449 

confidence:
  vs ret_1d: r=-0.0149, p=0.2898 
  vs ret_5d: r=+0.0054, p=0.7009 

clarity:
  vs ret_1d: r=-0.0068, p=0.6268 
  vs ret_5d: r=+0.0009, p=0.9465 

Direction 분위수별 평균 수익률
             ret_1d  ret_5d
direction_q                
Q1(부정)       -0.121   0.378
Q2           -0.002   0.590
Q3           -0.001   0.659
Q4           -0.134   0.246
Q5(긍정)       -0.054   0.417

Q5-Q1 스프레드 (1일): +0.067%p


In [22]:
# =============================================================================
# 셀 8-1: t+10 수익률로 재검증
# 목적: 뉴스 반영 래그 고려한 예측력 테스트
# =============================================================================

# 미래 수익률 붙이기 (t+1, t+5, t+10)
results = []

for _, row in daily_sentiment.iterrows():
    date = row["date"]
    ticker = row["ticker"]
    
    if ticker not in returns.columns:
        continue
    
    try:
        future_dates = returns.index[returns.index > date]
        
        # t+1
        ret_1d = returns.loc[future_dates[0], ticker] if len(future_dates) >= 1 else np.nan
        
        # t+5
        ret_5d = (1 + returns.loc[future_dates[:5], ticker]).prod() - 1 if len(future_dates) >= 5 else np.nan
        
        # t+10
        ret_10d = (1 + returns.loc[future_dates[:10], ticker]).prod() - 1 if len(future_dates) >= 10 else np.nan
            
        results.append({
            **row.to_dict(),
            "ret_1d": ret_1d,
            "ret_5d": ret_5d,
            "ret_10d": ret_10d
        })
    except:
        continue

df = pd.DataFrame(results)
df = df.dropna(subset=["ret_1d", "ret_5d", "ret_10d"])

print(f"분석 가능 데이터: {len(df)}건")

# =============================================================================
# 상관관계 분석 (t+1, t+5, t+10)
# =============================================================================
print("\n" + "="*60)
print("감성 지표 vs 미래 수익률 상관관계")
print("="*60)

for ret_col in ["ret_1d", "ret_5d", "ret_10d"]:
    corr, p = stats.pearsonr(df["direction"], df[ret_col])
    sig = "**" if p < 0.01 else "*" if p < 0.05 else ""
    print(f"direction vs {ret_col}: r={corr:+.4f}, p={p:.4f} {sig}")

# =============================================================================
# 분위수별 수익률 (t+10 추가)
# =============================================================================
print("\n" + "="*60)
print("Direction 분위수별 평균 수익률 (%)")
print("="*60)

df["direction_q"] = pd.qcut(df["direction"], q=5, labels=["Q1(부정)", "Q2", "Q3", "Q4", "Q5(긍정)"])

quintile_ret = df.groupby("direction_q")[["ret_1d", "ret_5d", "ret_10d"]].mean() * 100

print(quintile_ret.round(3))

# 스프레드
q5 = df[df["direction_q"] == "Q5(긍정)"]
q1 = df[df["direction_q"] == "Q1(부정)"]

print(f"\nQ5-Q1 스프레드:")
print(f"  1일:  {(q5['ret_1d'].mean() - q1['ret_1d'].mean())*100:+.3f}%p")
print(f"  5일:  {(q5['ret_5d'].mean() - q1['ret_5d'].mean())*100:+.3f}%p")
print(f"  10일: {(q5['ret_10d'].mean() - q1['ret_10d'].mean())*100:+.3f}%p")

분석 가능 데이터: 2799건

감성 지표 vs 미래 수익률 상관관계
direction vs ret_1d: r=-0.0149, p=0.4295 
direction vs ret_5d: r=-0.0383, p=0.0429 *
direction vs ret_10d: r=-0.0153, p=0.4193 

Direction 분위수별 평균 수익률 (%)
             ret_1d  ret_5d  ret_10d
direction_q                         
Q1(부정)        0.235   0.689    0.693
Q2            0.249   1.018    1.297
Q3            0.476   1.429    1.616
Q4            0.148   0.281    0.270
Q5(긍정)        0.219   0.398    0.758

Q5-Q1 스프레드:
  1일:  -0.016%p
  5일:  -0.291%p
  10일: +0.064%p
